# 07 - Placebo Tests

Two types of placebo: (1) fake treatment dates for NYC, (2) fake treated cities from the control group. Both should produce near-zero effects if the design is sound.

In [ ]:
# Data setup
# Set DATA_FILE to 'city_month_panel_real.parquet' after running build_real_panel.py
DATA_FILE = "city_month_panel_real.parquet"   # real Inside Airbnb data
# DATA_FILE = "city_month_panel.parquet"       # synthetic fallback (not committed)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

panel = pd.read_parquet(DATA_DIR / DATA_FILE)
panel["month"] = pd.to_datetime(panel["month"])

regs = pd.read_csv("../data/regulations.csv", parse_dates=["enforcement_date"])

print(f"Panel: {panel.shape}  |  Cities: {sorted(panel['city'].unique())}")
print(f"Date range: {panel['month'].min().date()} to {panel['month'].max().date()}")

In [ ]:
from linearmodels.panel import PanelOLS

def run_twfe_did(data, treat_city, treat_date, outcome="log_listings"):
    d = data.copy()
    d["post"]    = (d["month"] >= treat_date).astype(float)
    d["treated"] = (d["city"] == treat_city).astype(float)
    d["did"]     = d["treated"] * d["post"]
    idx = d.set_index(["city","month"])
    fe  = PanelOLS(idx[outcome], idx[["did"]],
                   entity_effects=True, time_effects=True).fit(
                   cov_type="clustered", cluster_entity=True)
    return fe.params["did"], fe.conf_int().loc["did"]


## Placebo 1 - Fake treatment dates for NYC (pre-period)

In [ ]:
nyc_data = panel[panel["city"] != "Florence"].copy()
TRUE_DATE = pd.Timestamp("2023-09-01")

placebo_dates = pd.date_range("2021-06-01", "2022-12-01", freq="3MS")
print(f"{'Placebo date':<15}  {'ATT':>8}  {'sig?':>6}")
print("-" * 35)
for d in placebo_dates:
    b, ci = run_twfe_did(nyc_data, "New York City", d)
    sig   = "*" if (ci["lower"] > 0 or ci["upper"] < 0) else ""
    print(f"{str(d.date()):<15}  {b:>8.4f}  {sig:>6}")

print()
b_true, _ = run_twfe_did(nyc_data, "New York City", TRUE_DATE)
print(f"True date (Sep 2023):  {b_true:>8.4f}  ** (true effect)")
print("Placebo ATTs near zero and insignificant => timing-specific effect")

## Placebo 2 - Fake treated cities

In [ ]:
controls = ["Amsterdam","Lisbon","Vienna","Barcelona"]
nyc_treat_date = pd.Timestamp("2023-09-01")
never_treated  = panel[panel["city"].isin(controls)].copy()

print(f"{'Placebo treated city':<20}  {'ATT':>8}  {'sig?':>6}")
print("-" * 38)
for city in controls:
    donor_data = never_treated.copy()
    b, ci = run_twfe_did(donor_data, city, nyc_treat_date)
    sig   = "*" if (ci["lower"] > 0 or ci["upper"] < 0) else ""
    print(f"{city:<20}  {b:>8.4f}  {sig:>6}")

print()
print(f"NYC (true treated):     {b_true:>8.4f}  ** (true effect)")
print("Placebo ATTs near zero => effect is specific to NYC, not a pan-city trend")